In [ ]:
from datasets import load_from_disk
import os
import pandas as pd
import textdistance
from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem import rdFingerprintGenerator
from typing import List, Dict
import numpy as np
from rdkit import Chem
# ignore RDKit warnings
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
# ignore the warnings from rdkit
# results = load_from_disk('results/finetune_metric/C&H/evaluation_finetune_1000_10_1_rmse_82')

def canonicalize_smiles(smiles: str) -> str:
    """Convert a SMILES string to its canonical form."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)

def evaluation(results, evaluation_features=['candidates'], result_len=5000):

    acc_dict = {'nmr2mol': {'Top 1': 0, 'Top 3': 0, 'Top 5': 0, 'Top 10': 0},}
    tan_dict = {'nmr2mol': {'Tani@1': [], 'Tani@3': [], 'Tani@5': [], 'Tani@10': []},}
    
    for feature in evaluation_features:
        assert feature in results.column_names, f"Feature '{feature}' not found in results."
        acc_dict[feature] = {'Top 1': 0, 'Top 3': 0, 'Top 5': 0, 'Top 10': 0}
        tan_dict[feature] = {'Tani@1': [], 'Tani@3': [], 'Tani@5': [], 'Tani@10': []}
    def canonicalize_dataset_smiles(entry, evaluation_features):
        entry['smiles'] = canonicalize_smiles(entry['smiles'])
        entry['original_pred_smiles'] = [canonicalize_smiles(s) for s in entry['original_pred_smiles']]
        for feature in evaluation_features:
            entry[feature] = [{'smiles': canonicalize_smiles(c['smiles']), **{k: v for k, v in c.items() if k != 'smiles'}} for c in entry[feature]]
        return entry
    results = results.map(lambda x: canonicalize_dataset_smiles(x, evaluation_features), num_proc=16)
    for entry in results:
        target_smiles = entry['smiles']
        initial_pred_smiles = entry['original_pred_smiles']
        for k in acc_dict['nmr2mol'].keys():
            top_k = int(k.split(' ')[1])
            if target_smiles in initial_pred_smiles[:top_k]:
                acc_dict['nmr2mol'][k] += 1
        for feature in evaluation_features:
            candidate_smiles = [c['smiles'] for c in entry[feature]]
            for k in acc_dict[feature].keys():
                top_k = int(k.split(' ')[1])
                if target_smiles in candidate_smiles[:top_k]:
                    acc_dict[feature][k] += 1
    acc_dict = {feature: {k: round(v / result_len, 4) for k, v in acc_dict[feature].items()} for feature in acc_dict}
    def cal_entry_tanimoto_similarity(entry, evaluation_features):
        target_smiles = entry['smiles']
        initial_pred_smiles = entry['original_pred_smiles']
        tan_scores = [cal_tanimoto_similarity(target_smiles, pred_smiles) for pred_smiles in initial_pred_smiles[:10]]
        entry['nmr2mol_tan_scores'] = tan_scores
        for feature in evaluation_features:
            for i in range(len(entry[feature])):
                candidate_smiles = entry[feature][i]['smiles']
                tan_score = cal_tanimoto_similarity(target_smiles, candidate_smiles)
                entry[feature][i]['tan_score'] = tan_score
        return entry
    results = results.map(lambda x: cal_entry_tanimoto_similarity(x, evaluation_features), num_proc=16)
    for entry in results:
        for k in tan_dict['nmr2mol'].keys():
            top_k = int(k.split('@')[1])
            tan_dict['nmr2mol'][k].append(max(entry['nmr2mol_tan_scores'][:top_k]))
        for feature in evaluation_features:
            tan_scores = [c['tan_score'] for c in entry[feature]]
            for k in tan_dict[feature].keys():
                top_k = int(k.split('@')[1])
                tan_dict[feature][k].append(max(tan_scores[:top_k]))
    tan_dict = {feature: {k: round(sum(v)/result_len, 4) for k, v in tan_dict[feature].items()} for feature in tan_dict}
    performance_dict = {}
    for feature in acc_dict.keys():
        performance_dict[feature] = {
            'Top 1': acc_dict[feature]['Top 1'],
            'Top 3': acc_dict[feature]['Top 3'],
            'Top 5': acc_dict[feature]['Top 5'],
            'Top 10': acc_dict[feature]['Top 10'],
            'Tani@1': tan_dict[feature]['Tani@1'],
            'Tani@3': tan_dict[feature]['Tani@3'],
            'Tani@5': tan_dict[feature]['Tani@5'],
            'Tani@10': tan_dict[feature]['Tani@10'],
        }
    return performance_dict

def levenshtein_similarity(truth: List[str], pred: List[str]) -> float:
    assert len(truth) == len(pred)
    scores = [
        textdistance.levenshtein.normalized_similarity(t, p)
        for t, p in zip(truth, pred)
    ]
    return scores

def cal_tanimoto_similarity(smiles_1: str, smiles_2: str) -> float:
    try:
        mol_1 = Chem.MolFromSmiles(smiles_1)
        mol_2 = Chem.MolFromSmiles(smiles_2)
        if mol_1 is None or mol_2 is None:
            return 0.0
        gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
        fp_1 = gen.GetFingerprint(mol_1)
        fp_2 = gen.GetFingerprint(mol_2)
        score = DataStructs.TanimotoSimilarity(fp_1, fp_2)
    except Exception as e:
        score = 0.0
    return score

# Metric

In [ ]:
results_dir_path = '../../results'
analysis_table_path = './tables'
os.makedirs(analysis_table_path, exist_ok = True)
metric_ratio_setting_mapping = {'811': '(0.8, 0.1, 0.1)', '721': '(0.7, 0.2, 0.1)',  '712': '(0.7, 0.1, 0.2)', 
                                '613': '(0.6, 0.1, 0.3)', '316': '(0.3, 0.1, 0.6)', '217': '(0.2, 0.1, 0.7)', 
                                '631': '(0.6, 0.3, 0.1)', '1000': '(1, 0, 0)', '0100': '(0, 1, 0)', '0010': '(0, 0, 1)', 
                                '820': '(0.8, 0.2, 0)', '640': '(0.6, 0.4, 0)', '460': '(0.4, 0.6, 0)', '280': '(0.2, 0.8, 0)',
                                '100': '(1, 0)', '010': '(0, 1)', '82': '(0.8, 0.2)', '64': '(0.6, 0.4)', 
                                '55': '(0.5, 0.5)', '46': '(0.4, 0.6)', '28': '(0.2, 0.8)'}

for experiment_type in ['NMRBank/metric-10']:
    os.makedirs(os.path.join(analysis_table_path, experiment_type), exist_ok = True)
    for modality in ['C', 'C&H', 'H']:
        df_list = []
        for i in range(5):
            df_list.append(pd.DataFrame(columns = ['source', 'NMR Similarity Metric', 'Ratio Setting', 'Top 1', 'Top 3', 'Top 5', 'Top 10', 'Tani@1', 'Tani@3', 'Tani@5', 'Tani@10']))
        df = pd.DataFrame(columns = ['source', 'NMR Similarity Metric', 'Ratio Setting', 'Top 1', 'Top 3', 'Top 5', 'Top 10', 'Tani@1', 'Tani@3', 'Tani@5', 'Tani@10'])
        result_dataset_name_list = sorted(os.listdir(os.path.join(results_dir_path, experiment_type, modality)))
        print('Compute Accuracy Table for', experiment_type, modality)
        for result_dataset_name in result_dataset_name_list:
            print(result_dataset_name)
            result_dataset_path = os.path.join(results_dir_path, experiment_type, modality, result_dataset_name)
            dataset = load_from_disk(result_dataset_path)
            performance_dict = evaluation(dataset, evaluation_features = ['candidates', 'candidates_info_1', 'candidates_info_2', 'candidates_info_3', 'candidates_info_4', 'candidates_info_5'], result_len = 1000)
            print(result_dataset_name)
            for key in performance_dict.keys():
                print(key, performance_dict[key])
            metric_ratio = metric_ratio_setting_mapping[result_dataset_path.split('_')[-1]]
            if result_dataset_path.split('_')[-2] == 'rmse':
                metric_type = 'RMSE'
            else:
                metric_type = 'Set Match Score' 
            feature_name = 'candidates'
            df.loc[len(df)] = [result_dataset_name, metric_type, metric_ratio, performance_dict[feature_name]['Top 1'], performance_dict[feature_name]['Top 3'], performance_dict[feature_name]['Top 5'], performance_dict[feature_name]['Top 10'], performance_dict[feature_name]['Tani@1'], performance_dict[feature_name]['Tani@3'], performance_dict[feature_name]['Tani@5'], performance_dict[feature_name]['Tani@10']]
            for i in range(1, 6):
                feature_name = f'candidates_info_{i}'
                df_list[i-1].loc[len(df_list[i-1])] = [result_dataset_name, metric_type, metric_ratio, performance_dict[feature_name]['Top 1'], performance_dict[feature_name]['Top 3'], performance_dict[feature_name]['Top 5'], performance_dict[feature_name]['Top 10'], performance_dict[feature_name]['Tani@1'], performance_dict[feature_name]['Tani@3'], performance_dict[feature_name]['Tani@5'], performance_dict[feature_name]['Tani@10']]
        for i in range(5):
            df_list[i].loc[len(df_list[i])] = ['nmr2mol', '-', '-', performance_dict['nmr2mol']['Top 1'], performance_dict['nmr2mol']['Top 3'], performance_dict['nmr2mol']['Top 5'], performance_dict['nmr2mol']['Top 10'], performance_dict['nmr2mol']['Tani@1'], performance_dict['nmr2mol']['Tani@3'], performance_dict['nmr2mol']['Tani@5'], performance_dict['nmr2mol']['Tani@10']]
            df_list[i].to_csv(os.path.join(analysis_table_path, experiment_type, f'{modality}_{i+1}.csv'), index = False)
        df.loc[len(df)] = ['nmr2mol', '-', '-', performance_dict['nmr2mol']['Top 1'], performance_dict['nmr2mol']['Top 3'], performance_dict['nmr2mol']['Top 5'], performance_dict['nmr2mol']['Top 10'], performance_dict['nmr2mol']['Tani@1'], performance_dict['nmr2mol']['Tani@3'], performance_dict['nmr2mol']['Tani@5'], performance_dict['nmr2mol']['Tani@10']]
        df.to_csv(os.path.join(analysis_table_path, experiment_type, f'{modality}.csv'), index = False)

In [ ]:
analysis_table_path = './tables'

df_NMRBank_C_10 = pd.read_csv(os.path.join(analysis_table_path, 'NMRBank/metric-10', 'C.csv'))
df_NMRBank_H_10 = pd.read_csv(os.path.join(analysis_table_path, 'NMRBank/metric-10', 'H.csv'))
df_NMRBank_CH_10 = pd.read_csv(os.path.join(analysis_table_path, 'NMRBank/metric-10', 'C&H.csv'))

df_NMRExp_C_10 = pd.read_csv(os.path.join(analysis_table_path, 'NMRExp/metric-10', 'C.csv'))
df_NMRExp_H_10 = pd.read_csv(os.path.join(analysis_table_path, 'NMRExp/metric-10', 'H.csv'))
df_NMRExp_CH_10 = pd.read_csv(os.path.join(analysis_table_path, 'NMRExp/metric-10', 'C&H.csv'))

df_USPTO_NMR_C_10 = pd.read_csv(os.path.join(analysis_table_path, 'USPTO-NMR/metric-10', 'C.csv'))
df_USPTO_NMR_H_10 = pd.read_csv(os.path.join(analysis_table_path, 'USPTO-NMR/metric-10', 'H.csv'))
df_USPTO_NMR_CH_10 = pd.read_csv(os.path.join(analysis_table_path, 'USPTO-NMR/metric-10', 'C&H.csv'))

# sort the 5 ratio settings with highest top 1 accuracy
df_NMRBank_C_10 = df_NMRBank_C_10.sort_values(by='Top 1', ascending=False).head(3)
df_NMRBank_H_10 = df_NMRBank_H_10.sort_values(by='Top 1', ascending=False).head(3)
df_NMRBank_CH_10 = df_NMRBank_CH_10.sort_values(by='Top 1', ascending=False).head(3)

df_NMRExp_C_10 = df_NMRExp_C_10.sort_values(by='Top 1', ascending=False).head(3)
df_NMRExp_H_10 = df_NMRExp_H_10.sort_values(by='Top 1', ascending=False).head(3)
df_NMRExp_CH_10 = df_NMRExp_CH_10.sort_values(by='Top 1', ascending=False).head(3)

df_USPTO_NMR_C_10 = df_USPTO_NMR_C_10.sort_values(by='Top 1', ascending=False).head(3)
df_USPTO_NMR_H_10 = df_USPTO_NMR_H_10.sort_values(by='Top 1', ascending=False).head(3)
df_USPTO_NMR_CH_10 = df_USPTO_NMR_CH_10.sort_values(by='Top 1', ascending=False).head(3)

# Transformer Result Analysis

In [ ]:
from datasets import Dataset
from datasets import load_from_disk, Dataset
from typing import Tuple, List, Dict, Union
import regex as re


def tokenize_formula(formula: str) -> list:
    return ' '.join(re.findall("[A-Z][a-z]?|\d+|.", formula)) + ' '

def process_hnmr(multiplets: List[Dict[str, Union[str, float, int]]]) -> str:

    multiplet_str = "1HNMR "
    for peak in multiplets:
        range_max = float(peak["rangeMax"]) 
        range_min = float(peak["rangeMin"]) 

        formatted_peak = ""
        formatted_peak = formatted_peak + "{:.2f} {:.2f} ".format(range_max, range_min)        
        formatted_peak = formatted_peak +  "{} {}H ".format(
                                                            peak["category"],
                                                            peak["nH"],
                                                        )
        js = str(peak["j_values"])
        if js != "None":
            split_js = js.split("_")
            split_js = list(filter(None, split_js))
            processed_js = ["{:.2f}".format(float(j)) for j in split_js]
            formatted_js = "J " + " ".join(processed_js)
            formatted_peak += formatted_js

        multiplet_str += formatted_peak.strip() + " | "

    # Remove last separating token
    multiplet_str = multiplet_str[:-2]
    return multiplet_str

def process_cnmr(carbon_nmr: List[Dict[str, Union[str, float, int]]]) -> str:
    nmr_string = "13CNMR "
    for peak in carbon_nmr:
        nmr_string += str(round(float(peak["delta (ppm)"]), 1)) + " "

    return nmr_string

for dataset_type in ['NMRBank', 'NMRExp', 'USPTO-NMR']:
    test_set_10k = load_from_disk(f'../../data/{dataset_type}/{dataset_type}-10000')

    for modality in ['C', 'H', 'CH']:
        test_set_10k_source = []
        for entry in test_set_10k:
            tokenized_formula = tokenize_formula(entry['molecular_formula'])
            tokenized_input = tokenized_formula
            if modality == 'H' or modality == 'CH':
                h_nmr_string = process_hnmr(entry['h_nmr_peaks'])
                tokenized_input += h_nmr_string
            if modality == 'C' or modality == 'CH':
                c_nmr_string = process_cnmr(entry['c_nmr_peaks'])
                tokenized_input += c_nmr_string
            test_set_10k_source.append(tokenized_input.strip())
        with open(f'../../multimodal-spectroscopic-dataset/runs/{dataset_type}/{modality}/data/src-test.txt', 'r') as f:
            source_list = f.read().splitlines()
        with open(f'../../multimodal-spectroscopic-dataset/runs/{dataset_type}/{modality}/data/tgt-test.txt', 'r') as f:
            target_list = f.read().splitlines()
        with open(f'../../multimodal-spectroscopic-dataset/runs/{dataset_type}/{modality}/pred-test.txt', 'r') as f:
            pred_list = f.read().splitlines()
        data_list = []
        data_list_10000 = []
        for i in range(len(source_list)):
            data_list.append({'source': source_list[i], 'smiles': target_list[i].replace(' ', ''), 'original_pred_smiles': [p.replace(' ', '') for p in pred_list[i*10:(i+1)*10]]})
            if source_list[i] in test_set_10k_source:
                data_list_10000.append({'source': source_list[i], 'smiles': target_list[i].replace(' ', ''), 'original_pred_smiles': [p.replace(' ', '') for p in pred_list[i*10:(i+1)*10]]})
        dataset = Dataset.from_list(data_list)
        dataset_10000 = Dataset.from_list(data_list_10000)
        if modality == 'CH':
            modality = 'C&H'
        dataset.save_to_disk(f'../../results/{dataset_type}/transformer/{modality}')
        dataset_10000.save_to_disk(f'../../results/{dataset_type}/transformer/{modality}_10000')

In [ ]:
results_dir_path = '../../results'
analysis_table_path = './tables'
os.makedirs(analysis_table_path, exist_ok = True)
for experiment_type in ['NMRBank/transformer', 'NMRExp/transformer', 'USPTO-NMR/transformer']:
    df = pd.DataFrame(columns = ['Modality', 'Source', 'Top 1', 'Top 3', 'Top 5', 'Top 10', 'Tani@1', 'Tani@3', 'Tani@5', 'Tani@10'])
    print('Compute Accuracy Table for', experiment_type)
    for modality in ['C&H', 'C', 'H', 'C&H_10000', 'C_10000', 'H_10000']:
        result_dataset_path = os.path.join(results_dir_path, experiment_type, modality)
        print(result_dataset_path)
        dataset = load_from_disk(result_dataset_path)
        result_len = 10000 if '10000' in modality else len(dataset)
        performance_dict = evaluation(dataset, evaluation_features = [], result_len = result_len)
        for key in performance_dict.keys():
            print(key, performance_dict[key])
        feature_name = 'nmr2mol'
        df.loc[len(df)] = [modality, result_dataset_path, performance_dict[feature_name]['Top 1'], performance_dict[feature_name]['Top 3'], performance_dict[feature_name]['Top 5'], performance_dict[feature_name]['Top 10'], 
                                        performance_dict[feature_name]['Tani@1'], performance_dict[feature_name]['Tani@3'], performance_dict[feature_name]['Tani@5'], performance_dict[feature_name]['Tani@10']]
                
    df.to_csv(os.path.join(analysis_table_path, experiment_type, f'all.csv'), index = False)

# Full Test

In [ ]:
results_dir_path = '../../results'
analysis_table_path = './tables'
os.makedirs(analysis_table_path, exist_ok = True)
for experiment_type in ['NMRBank/10000', 'NMRExp/10000', 'USPTO-NMR/10000']:
    df_all_round = pd.DataFrame(columns = ['Modality', 'Source', 'Revision Round', 'Top 1', 'Top 3', 'Top 5', 'Top 10', 'Tani@1', 'Tani@3', 'Tani@5', 'Tani@10'])
    df_final_round = pd.DataFrame(columns = ['Modality', 'Source', 'Top 1', 'Top 3', 'Top 5', 'Top 10', 'Tani@1', 'Tani@3', 'Tani@5', 'Tani@10'])
    print('Compute Accuracy Table for', experiment_type)
    for modality in ['C&H', 'C', 'H']:
        result_dataset_name_list = os.listdir(os.path.join(results_dir_path, experiment_type, modality))
        for result_dataset_name in result_dataset_name_list:
            result_dataset_path = os.path.join(results_dir_path, experiment_type, modality, result_dataset_name)
            dataset = load_from_disk(result_dataset_path)
            performance_dict = evaluation(dataset, evaluation_features = ['candidates', 'candidates_info_1', 'candidates_info_2', 'candidates_info_3', 'candidates_info_4', 'candidates_info_5'], result_len = 10000)
            print(result_dataset_name)
            for key in performance_dict.keys():
                print(key, performance_dict[key])
            feature_name = 'candidates'
            df_final_round.loc[len(df_final_round)] = [modality, result_dataset_name, performance_dict[feature_name]['Top 1'], performance_dict[feature_name]['Top 3'], performance_dict[feature_name]['Top 5'], performance_dict[feature_name]['Top 10'], 
                                           performance_dict[feature_name]['Tani@1'], performance_dict[feature_name]['Tani@3'], performance_dict[feature_name]['Tani@5'], performance_dict[feature_name]['Tani@10']]
            for i in range(1, 6):
                feature_name = f'candidates_info_{i}'
                df_all_round.loc[len(df_all_round)] = [modality, result_dataset_name, i, performance_dict[feature_name]['Top 1'], performance_dict[feature_name]['Top 3'], performance_dict[feature_name]['Top 5'], performance_dict[feature_name]['Top 10'], 
                                           performance_dict[feature_name]['Tani@1'], performance_dict[feature_name]['Tani@3'], performance_dict[feature_name]['Tani@5'], performance_dict[feature_name]['Tani@10']]

        
        df_final_round.loc[len(df_final_round)] = [modality, 'nmr2mol', performance_dict['nmr2mol']['Top 1'], performance_dict['nmr2mol']['Top 3'], performance_dict['nmr2mol']['Top 5'], performance_dict['nmr2mol']['Top 10'],
                                           performance_dict['nmr2mol']['Tani@1'], performance_dict['nmr2mol']['Tani@3'], performance_dict['nmr2mol']['Tani@5'], performance_dict['nmr2mol']['Tani@10']]
        df_all_round.loc[len(df_all_round)] = [modality, 'nmr2mol', 0, performance_dict['nmr2mol']['Top 1'], performance_dict['nmr2mol']['Top 3'], performance_dict['nmr2mol']['Top 5'], performance_dict['nmr2mol']['Top 10'],
                                           performance_dict['nmr2mol']['Tani@1'], performance_dict['nmr2mol']['Tani@3'], performance_dict['nmr2mol']['Tani@5'], performance_dict['nmr2mol']['Tani@10']]
    
    df_final_round.to_csv(os.path.join(analysis_table_path, experiment_type, f'final.csv'), index = False)
    df_all_round.to_csv(os.path.join(analysis_table_path, experiment_type, f'all.csv'), index = False)

# Result Merge

In [ ]:
import json
df = pd.DataFrame(columns = ['Model', 'Dataset', 'Modality', 'Top 1', 'Top 5', 'Top 10', 'Tani@1', 'Tani@5', 'Tani@10'])

# merge the results from NMR-RISE
for dataset_type in ['NMRBank', 'NMRExp', 'USPTO-NMR']:
    df_nmr_rise = pd.read_csv(os.path.join('./tables', f'{dataset_type}/10000/final.csv'))
    for index, row in df_nmr_rise.iterrows():
        if row['Source'] == 'nmr2mol':
            model_name = 'NMR2Mol'
        else:
            model_name = 'NMR-RISE'
        modality = row['Modality']
        df.loc[len(df)] = [model_name, dataset_type, modality, round(100*row['Top 1'], 2), round(100*row['Top 5'], 2), round(100*row['Top 10'], 2), round(100*row['Tani@1'], 2), round(100*row['Tani@5'], 2), round(100*row['Tani@10'], 2)]

# merge the results from transformer
for dataset_type in ['NMRBank', 'NMRExp', 'USPTO-NMR']:
    df_transformer = pd.read_csv(os.path.join('./tables', f'{dataset_type}/transformer/all.csv'))
    for index, row in df_transformer.iterrows():
        modality = row['Modality']
        if '10000' not in modality:
            df.loc[len(df)] = ['Transformer', dataset_type, modality, round(100*row['Top 1'], 2), round(100*row['Top 5'], 2), round(100*row['Top 10'], 2), round(100*row['Tani@1'], 2), round(100*row['Tani@5'], 2), round(100*row['Tani@10'], 2)]

# merge the results from nmr-solver
for dataset_type in ['NMRBank', 'NMRExp', 'USPTO-NMR']:
    for modality in ['C', 'H', 'C&H']:
        with open(f'../../results_nmr-solver/{dataset_type}/{modality}/iter_2_formula.json', 'r') as f:
            result = json.load(f)
        df.loc[len(df)] = ['NMR-Solver', dataset_type, modality, round(100*result['recall@1'], 2), round(100*result['recall@5'], 2), round(100*result['recall@10'], 2), round(100*result['tanimoto@1'], 2), round(100*result['tanimoto@5'], 2), round(100*result['tanimoto@10'], 2)]

In [ ]:
# sort df in Model (NMR2Mol, Transformer, NMR-RISE, NMR-Solver), Dataset (USPTO-NMR, NMRBank, NMRExp), Modality (C, H, C&H)
df.Model = pd.Categorical(df.Model, categories=['NMR2Mol', 'Transformer', 'NMR-RISE', 'NMR-Solver'], ordered=True)
df.Dataset = pd.Categorical(df.Dataset, categories=['USPTO-NMR', 'NMRBank', 'NMRExp'], ordered=True)
df.Modality = pd.Categorical(df.Modality, categories=['C', 'H', 'C&H'], ordered=True)
df = df.sort_values(by=['Model', 'Dataset', 'Modality'])
df.to_csv('./tables/final_comparison.csv', index = False)

# Inference Parameter

In [ ]:
results_dir_path = '../../results'
analysis_table_path = './tables'
os.makedirs(analysis_table_path, exist_ok = True)
for experiment_type in ['NMRExp/param']:
    df = pd.DataFrame(columns = ['Modality', 'Source', 'Beam Size', 'Revision_Rounds', 'Top 1', 'Top 3', 'Top 5', 'Top 10', 'Tani@1', 'Tani@3', 'Tani@5', 'Tani@10'])
    print('Compute Accuracy Table for', experiment_type)
    for modality in ['C&H', 'C', 'H']:
        result_dataset_name_list = os.listdir(os.path.join(results_dir_path, experiment_type, modality))
        result_dataset_name_list = sorted(result_dataset_name_list, key = lambda x: int(x.split('_')[-1]))
        for result_dataset_name in result_dataset_name_list:
            result_dataset_path = os.path.join(results_dir_path, experiment_type, modality, result_dataset_name)
            dataset = load_from_disk(result_dataset_path)
            acc_dict = evaluation(dataset, evaluation_features = ['candidates', 'candidates_info_1', 'candidates_info_2', 'candidates_info_3', 'candidates_info_4', 'candidates_info_5'], result_len = 1000)
            print(result_dataset_name)
            for key in acc_dict.keys():
                print(key, acc_dict[key])
            for feature_name in ['candidates_info_1', 'candidates_info_2', 'candidates_info_3', 'candidates_info_4', 'candidates_info_5']:
                beam_size = int(result_dataset_name.split('_')[3])
                revision_rounds = int(feature_name.split('_')[-1])
                df.loc[len(df)] = [modality, result_dataset_name, beam_size, revision_rounds, acc_dict[feature_name]['Top 1'], acc_dict[feature_name]['Top 3'], acc_dict[feature_name]['Top 5'], acc_dict[feature_name]['Top 10'], acc_dict[feature_name]['Tani@1'], acc_dict[feature_name]['Tani@3'], acc_dict[feature_name]['Tani@5'], acc_dict[feature_name]['Tani@10']]
            df.loc[len(df)] = [modality, 'nmr2mol', '-', '-', acc_dict['nmr2mol']['Top 1'], acc_dict['nmr2mol']['Top 3'], acc_dict['nmr2mol']['Top 5'], acc_dict['nmr2mol']['Top 10'], acc_dict['nmr2mol']['Tani@1'], acc_dict['nmr2mol']['Tani@3'], acc_dict['nmr2mol']['Tani@5'], acc_dict['nmr2mol']['Tani@10']]
    df.to_csv(os.path.join(analysis_table_path, f'{experiment_type}.csv'), index = False)

# MolRef

In [ ]:
results_dir_path = '../../results'
analysis_table_path = './tables'
os.makedirs(analysis_table_path, exist_ok = True)
for experiment_type in ['NMRBank/molref', 'NMRExp/molref']:
    df = pd.DataFrame(columns = ['Modality', 'Source', 'Augmentation Scale', 'Top 1', 'Top 3', 'Top 5', 'Top 10', 'Tani@1', 'Tani@3', 'Tani@5', 'Tani@10'])
    print('Compute Accuracy Table for', experiment_type)
    for modality in ['C&H', 'C', 'H']:
        result_dataset_name_list = os.listdir(os.path.join(results_dir_path, experiment_type, modality))
        result_dataset_name_list = sorted(result_dataset_name_list, key = lambda x: int(x.split('_')[-1]))
        for result_dataset_name in result_dataset_name_list:
            result_dataset_path = os.path.join(results_dir_path, experiment_type, modality, result_dataset_name)
            dataset = load_from_disk(result_dataset_path)
            acc_dict = evaluation(dataset, evaluation_features = ['candidates', 'candidates_info_1'], result_len = 1000)
            print(result_dataset_name)
            for key in acc_dict.keys():
                print(key, acc_dict[key])
            feature_name = 'candidates_info_1'
            augmentation_scale = int(result_dataset_name.split('_')[-1])
            df.loc[len(df)] = [modality, result_dataset_name, augmentation_scale, acc_dict[feature_name]['Top 1'], acc_dict[feature_name]['Top 3'], acc_dict[feature_name]['Top 5'], acc_dict[feature_name]['Top 10'], acc_dict[feature_name]['Tani@1'], acc_dict[feature_name]['Tani@3'], acc_dict[feature_name]['Tani@5'], acc_dict[feature_name]['Tani@10']]
        feature_name = 'candidates_info_0'
        df.loc[len(df)] = [modality, 'nmr2mol', '-', acc_dict['nmr2mol']['Top 1'], acc_dict['nmr2mol']['Top 3'], acc_dict['nmr2mol']['Top 5'], acc_dict['nmr2mol']['Top 10'], acc_dict['nmr2mol']['Tani@1'], acc_dict['nmr2mol']['Tani@3'], acc_dict['nmr2mol']['Tani@5'], acc_dict['nmr2mol']['Tani@10']]
    df.to_csv(os.path.join(analysis_table_path, f'{experiment_type}.csv'), index = False)

# Mol2NMR

In [ ]:
results_dir_path = '../../results'
analysis_table_path = './tables'
os.makedirs(analysis_table_path, exist_ok = True)
for experiment_type in ['NMRExp/mol2nmr']:
    df = pd.DataFrame(columns = ['Modality', 'Source', 'Iteration', 'Top 1', 'Top 3', 'Top 5', 'Top 10', 'Tani@1', 'Tani@3', 'Tani@5', 'Tani@10'])
    print('Compute Accuracy Table for', experiment_type)
    for modality in ['C&H', 'C', 'H']:
        result_dataset_name_list = os.listdir(os.path.join(results_dir_path, experiment_type, modality))
        for result_dataset_name in result_dataset_name_list:
            result_dataset_path = os.path.join(results_dir_path, experiment_type, modality, result_dataset_name)
            dataset = load_from_disk(result_dataset_path)
            acc_dict = evaluation(dataset, evaluation_features = ['candidates', 'candidates_info_1'], result_len = 1000)
            print(result_dataset_name)
            for key in acc_dict.keys():
                print(key, acc_dict[key])
            feature_name = 'candidates_info_1'
            iteration = int(result_dataset_name.split('_')[-1]) if result_dataset_name.split('_')[-1].isdigit() else result_dataset_name.split('_')[-1]
            df.loc[len(df)] = [modality, result_dataset_name, iteration, acc_dict[feature_name]['Top 1'], acc_dict[feature_name]['Top 3'], acc_dict[feature_name]['Top 5'], acc_dict[feature_name]['Top 10'], acc_dict[feature_name]['Tani@1'], acc_dict[feature_name]['Tani@3'], acc_dict[feature_name]['Tani@5'], acc_dict[feature_name]['Tani@10']]
        df.loc[len(df)] = [modality, 'nmr2mol', '-', acc_dict['nmr2mol']['Top 1'], acc_dict['nmr2mol']['Top 3'], acc_dict['nmr2mol']['Top 5'], acc_dict['nmr2mol']['Top 10'], acc_dict['nmr2mol']['Tani@1'], acc_dict['nmr2mol']['Tani@3'], acc_dict['nmr2mol']['Tani@5'], acc_dict['nmr2mol']['Tani@10']]
    df.to_csv(os.path.join(analysis_table_path, f'{experiment_type}.csv'), index = False)

In [ ]:
from nmr_rise.utils.data import conformation_construction, peak_match, hnmr_list_generation, set2vec, euclidean_distance, set_match_score_weighted
import numpy as np
from datasets import load_from_disk
import os
import pandas as pd

def calculate_rmse(matched_data):
    """Calculate RMSE between experimental and predicted chemical shifts"""
    
    # Calculate RMSE
    squared_diff = []
    for v in matched_data.values():
        squared_diff.append((v['exp_peak'] - v['pred_peak'])**2)

    # squared_diff = [(v['exp_peak'] - v['pred_peak'])**2 for v in matched_data.values]
    rmse = np.sqrt(np.mean(squared_diff))
    # return rmse with 2 decimal places
    return rmse

def entry_metric_calculation(entry):
    hnmr_list = hnmr_list_generation(entry['h_nmr_peaks'])
    cnmr_list = [cnmr['delta (ppm)'] for cnmr in entry['c_nmr_peaks']]
    atoms = entry['atoms']
    pred_nmr = entry['predict']

    hnmr_pred = [pred_nmr[i] for i in range(len(atoms)) if atoms[i] == 'H']
    cnmr_pred = [pred_nmr[i] for i in range(len(atoms)) if atoms[i] == 'C']
    if hnmr_list and hnmr_pred:
        hnmr_match = peak_match(hnmr_list, hnmr_pred)
        entry['hnmr_rmse'] = calculate_rmse(hnmr_match)
        entry['hnmr_rmse_normalized'] = np.exp(-entry['hnmr_rmse'] / 0.5)
    else:
        entry['hnmr_rmse'] = None
        entry['hnmr_rmse_normalized'] = None
    
    if cnmr_list and cnmr_pred:
        cnmr_match = peak_match(cnmr_list, cnmr_pred)
        entry['cnmr_rmse'] = calculate_rmse(cnmr_match)
        entry['cnmr_rmse_normalized'] = np.exp(-entry['cnmr_rmse'] / 5)
    else:
        entry['cnmr_rmse'] = None
        entry['cnmr_rmse_normalized'] = None
    entry['cnmr_vec_sim'] = euclidean_distance(set2vec(np.array(cnmr_list), nmr_type='C'), set2vec(np.array(sorted(cnmr_pred)), nmr_type='C'))
    entry['hnmr_vec_sim'] = euclidean_distance(set2vec(np.array(hnmr_list), nmr_type='H'), set2vec(np.array(sorted(hnmr_pred)), nmr_type='H'))

    entry['cnmr_set_match_score'] = set_match_score_weighted(np.array(cnmr_list), np.array(cnmr_pred), sigma=10)
    entry['hnmr_set_match_score'] = set_match_score_weighted(np.array(hnmr_list), np.array(hnmr_pred), sigma=1)

    return entry


for data_type in ['NMRBank', 'NMRExp', 'USPTO-NMR']:
    df = pd.DataFrame(columns=['iteration', 'cnmr_rmse', 'hnmr_rmse', 'cnmr_rmse_normalized', 'hnmr_rmse_normalized', 'cnmr_vec_sim', 'hnmr_vec_sim', 'cnmr_set_match_score', 'hnmr_set_match_score'])

    dataset_dir_path = f'../../data/{data_type}/mol2nmr/'
    dataset_name_list = os.listdir(dataset_dir_path)
    
    for dataset_dir in dataset_name_list:
        if dataset_dir == 'full_cc':
            continue
        dataset = load_from_disk(f'{dataset_dir_path}/{dataset_dir}')
        test_dataset = dataset['test'].map(entry_metric_calculation, num_proc=os.cpu_count())
        # train_dataset = dataset['train'].map(entry_metric_calculation, num_proc=os.cpu_count())
        metrics = {
        'cnmr_rmse': round(float(np.mean([e for e in test_dataset['cnmr_rmse'] if e is not None])), 4),
        'hnmr_rmse': round(float(np.mean([e for e in test_dataset['hnmr_rmse'] if e is not None])), 4),
        'cnmr_rmse_normalized': round(float(np.mean([e for e in test_dataset['cnmr_rmse_normalized'] if e is not None])), 4),
        'hnmr_rmse_normalized': round(float(np.mean([e for e in test_dataset['hnmr_rmse_normalized'] if e is not None])), 4),
        'cnmr_vec_sim': round(float(np.mean(test_dataset['cnmr_vec_sim'])), 4),
        'hnmr_vec_sim': round(float(np.mean(test_dataset['hnmr_vec_sim'])), 4),
        'cnmr_set_match_score': round(float(np.mean(test_dataset['cnmr_set_match_score'])), 4),
        'hnmr_set_match_score': round(float(np.mean(test_dataset['hnmr_set_match_score'])), 4),
        }
        df.loc[len(df)] = {**{'iteration': int(dataset_dir.split('_')[-1])}, **metrics}
    
    df.to_csv(f'tables/{data_type}/mol2nmr_metric_summary.csv', index=False)
